# Policy Comparison

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/antimicrobial-resistance=/path/to/antimicrobial-resistance"` at the top of any cell.

Compare four prescribing policies for antimicrobial stewardship using **10 stochastic trajectories** per policy to quantify uncertainty:
- **Baseline**: constant cephalosporin rate (0.3)
- **Cycling**: quarterly alternation between high (0.3) and low (0.05) rates
- **Threshold**: adaptive switch to low rate when resistance exceeds 15%
- **Restriction**: linear ramp-down from 0.3 to 0.1 over 26 time steps

All simulations use the learned posterior mean parameters from Phase 3 inference. Plots show the **mean** trajectory (solid line) with **±2σ** bounds (dashed lines) across 10 seeds.

**Prerequisites:**
```bash
python3 dat/run_policy_evaluation.py  # runs 4 policies × 10 seeds = 40 simulations
```

In [ ]:
import (
	"fmt"
	"math"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	"github.com/umbralcalc/stochadex/pkg/simulator"
)

var policyNames = []string{"Baseline", "Cycling", "Threshold", "Restriction"}
var policyKeys = []string{"baseline", "cycling", "threshold", "restriction"}
const nSeeds = 10

// loadPolicySeeds loads all seed trajectories for a given policy.
func loadPolicySeeds(key string) []*simulator.StateTimeStorage {
	var storages []*simulator.StateTimeStorage
	for i := 0; i < nSeeds; i++ {
		path := fmt.Sprintf("../dat/policy_%s_seed%d.log", key, i)
		s, err := analysis.NewStateTimeStorageFromJsonLogEntries(path)
		if err != nil {
			fmt.Printf("Warning: could not load %s: %v\n", path, err)
			continue
		}
		storages = append(storages, s)
	}
	return storages
}

// multiSeedStats computes mean and std of a partition column across seeds.
func multiSeedStats(storages []*simulator.StateTimeStorage, partition string, col int) (times, mean, std []float64) {
	if len(storages) == 0 {
		return
	}
	times = storages[0].GetTimes()
	n := len(times)
	mean = make([]float64, n)
	std = make([]float64, n)

	allVals := make([][]float64, len(storages))
	for s, st := range storages {
		ref := analysis.DataRef{PartitionName: partition, ValueIndices: []int{col}}
		allVals[s] = ref.GetFromStorage(st)[0]
	}

	for i := 0; i < n; i++ {
		sum := 0.0
		for s := range storages {
			sum += allVals[s][i]
		}
		mean[i] = sum / float64(len(storages))

		sumSq := 0.0
		for s := range storages {
			d := allVals[s][i] - mean[i]
			sumSq += d * d
		}
		std[i] = math.Sqrt(sumSq / float64(len(storages)))
	}
	return
}

// xyLineData builds []opts.LineData with [x, y] pairs.
func xyLineData(xs []float64, ys []float64) []opts.LineData {
	data := make([]opts.LineData, len(xs))
	for i := range xs {
		data[i] = opts.LineData{Value: []interface{}{xs[i], ys[i]}}
	}
	return data
}

%%

// Verify all seed files load
for i, key := range policyKeys {
	storages := loadPolicySeeds(key)
	fmt.Printf("  %s: %d/%d seeds loaded\n", policyNames[i], len(storages), nSeeds)
}

## Prescribing Rate

How each policy modulates the cephalosporin prescribing rate over time.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

// Prescribing rate is deterministic — just load seed 0 for each policy
storages := make(map[string]*simulator.StateTimeStorage)
for i, key := range policyKeys {
	seeds := loadPolicySeeds(key)
	if len(seeds) > 0 {
		storages[policyNames[i]] = seeds[0]
	}
}

line := analysis.NewLinePlotFromPartition(
	storages["Baseline"],
	analysis.DataRef{PartitionName: "prescribing", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "prescribing", ValueIndices: []int{0}},
	},
	nil,
)

for _, name := range policyNames[1:] {
	s := storages[name]
	times := s.GetTimes()
	ref := analysis.DataRef{PartitionName: "prescribing", ValueIndices: []int{0}}
	vals := ref.GetFromStorage(s)[0]
	line.AddSeries(name, xyLineData(times, vals))
}

line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Cephalosporin prescribing rate by policy",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Prescribing rate"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
)
gonb_echarts.Display(line, "width: 1024px; height: 400px; background: white;")

## Resistance Ratio

The fraction of colonised patients carrying the resistant strain: R / (S + R). This is the primary outcome — lower is better.

In [ ]:
import (
	"math"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

// resistanceRatioStats computes mean and std of R/(S+R) across seeds.
func resistanceRatioStats(storages []*simulator.StateTimeStorage) (times, mean, std []float64) {
	if len(storages) == 0 {
		return
	}
	times = storages[0].GetTimes()
	n := len(times)

	allRatios := make([][]float64, len(storages))
	for s, st := range storages {
		sRef := analysis.DataRef{PartitionName: "colonisation", ValueIndices: []int{0}}
		rRef := analysis.DataRef{PartitionName: "colonisation", ValueIndices: []int{1}}
		sVals := sRef.GetFromStorage(st)[0]
		rVals := rRef.GetFromStorage(st)[0]
		ratios := make([]float64, n)
		for i := range sVals {
			if sVals[i]+rVals[i] > 0 {
				ratios[i] = rVals[i] / (sVals[i] + rVals[i])
			}
		}
		allRatios[s] = ratios
	}

	mean = make([]float64, n)
	std = make([]float64, n)
	for i := 0; i < n; i++ {
		sum := 0.0
		for s := range storages {
			sum += allRatios[s][i]
		}
		mean[i] = sum / float64(len(storages))
		sumSq := 0.0
		for s := range storages {
			d := allRatios[s][i] - mean[i]
			sumSq += d * d
		}
		std[i] = math.Sqrt(sumSq / float64(len(storages)))
	}
	return
}

%%

var colors = []string{"#5470c6", "#91cc75", "#fac858", "#ee6666"}

line := charts.NewLine()
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Resistance ratio R/(S+R) by policy (mean ± 2σ, 10 seeds)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Resistance ratio"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)

for i, key := range policyKeys {
	storages := loadPolicySeeds(key)
	times, mean, std := resistanceRatioStats(storages)
	upper := make([]float64, len(mean))
	lower := make([]float64, len(mean))
	for j := range mean {
		upper[j] = mean[j] + 2*std[j]
		lower[j] = mean[j] - 2*std[j]
	}
	line.AddSeries(policyNames[i], xyLineData(times, mean),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 2.5, Color: colors[i]}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i]}),
	)
	line.AddSeries(policyNames[i]+" +2σ", xyLineData(times, upper),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
	line.AddSeries(policyNames[i]+" −2σ", xyLineData(times, lower),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
}
gonb_echarts.Display(line, "width: 1024px; height: 450px; background: white;")

## Colonisation Dynamics

Susceptible and resistant colonisation fractions under each policy.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

var colors = []string{"#5470c6", "#91cc75", "#fac858", "#ee6666"}

// Susceptible fraction: mean ± 2σ
susLine := charts.NewLine()
susLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Susceptible colonisation fraction (mean ± 2σ, 10 seeds)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Susceptible fraction"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)
for i, key := range policyKeys {
	storages := loadPolicySeeds(key)
	times, mean, std := multiSeedStats(storages, "colonisation", 0)
	upper := make([]float64, len(mean))
	lower := make([]float64, len(mean))
	for j := range mean {
		upper[j] = mean[j] + 2*std[j]
		lower[j] = mean[j] - 2*std[j]
	}
	susLine.AddSeries(policyNames[i], xyLineData(times, mean),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 2.5, Color: colors[i]}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i]}),
	)
	susLine.AddSeries(policyNames[i]+" +2σ", xyLineData(times, upper),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
	susLine.AddSeries(policyNames[i]+" −2σ", xyLineData(times, lower),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
}
gonb_echarts.Display(susLine, "width: 1024px; height: 400px; background: white;")

// Resistant fraction: mean ± 2σ
resLine := charts.NewLine()
resLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Resistant colonisation fraction (mean ± 2σ, 10 seeds)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Resistant fraction"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)
for i, key := range policyKeys {
	storages := loadPolicySeeds(key)
	times, mean, std := multiSeedStats(storages, "colonisation", 1)
	upper := make([]float64, len(mean))
	lower := make([]float64, len(mean))
	for j := range mean {
		upper[j] = mean[j] + 2*std[j]
		lower[j] = mean[j] - 2*std[j]
	}
	resLine.AddSeries(policyNames[i], xyLineData(times, mean),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 2.5, Color: colors[i]}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i]}),
	)
	resLine.AddSeries(policyNames[i]+" +2σ", xyLineData(times, upper),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
	resLine.AddSeries(policyNames[i]+" −2σ", xyLineData(times, lower),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
}
gonb_echarts.Display(resLine, "width: 1024px; height: 400px; background: white;")

## Cumulative Resistant BSI

Cumulative count of resistant bloodstream infections over time — the key clinical outcome for comparing stewardship policies.

In [ ]:
import (
	"fmt"
	"math"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

// cumulativeBSIStats computes mean and std of cumulative resistant BSI across seeds.
func cumulativeBSIStats(storages []*simulator.StateTimeStorage) (times, mean, std []float64) {
	if len(storages) == 0 {
		return
	}
	times = storages[0].GetTimes()
	n := len(times)

	allCum := make([][]float64, len(storages))
	for s, st := range storages {
		ref := analysis.DataRef{PartitionName: "infection", ValueIndices: []int{1}}
		bsiR := ref.GetFromStorage(st)[0]
		cum := make([]float64, n)
		sum := 0.0
		for i, v := range bsiR {
			sum += v
			cum[i] = sum
		}
		allCum[s] = cum
	}

	mean = make([]float64, n)
	std = make([]float64, n)
	for i := 0; i < n; i++ {
		sum := 0.0
		for s := range storages {
			sum += allCum[s][i]
		}
		mean[i] = sum / float64(len(storages))
		sumSq := 0.0
		for s := range storages {
			d := allCum[s][i] - mean[i]
			sumSq += d * d
		}
		std[i] = math.Sqrt(sumSq / float64(len(storages)))
	}
	return
}

%%

var colors = []string{"#5470c6", "#91cc75", "#fac858", "#ee6666"}

line := charts.NewLine()
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Cumulative resistant BSI (mean ± 2σ, 10 seeds)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Cumulative resistant BSI"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)

for i, key := range policyKeys {
	storages := loadPolicySeeds(key)
	times, mean, std := cumulativeBSIStats(storages)
	upper := make([]float64, len(mean))
	lower := make([]float64, len(mean))
	for j := range mean {
		upper[j] = mean[j] + 2*std[j]
		lower[j] = mean[j] - 2*std[j]
	}
	line.AddSeries(policyNames[i], xyLineData(times, mean),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 2.5, Color: colors[i]}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i]}),
	)
	line.AddSeries(policyNames[i]+" +2σ", xyLineData(times, upper),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
	line.AddSeries(policyNames[i]+" −2σ", xyLineData(times, lower),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 1, Color: colors[i], Type: "dashed", Opacity: opts.Float(0.5)}),
		charts.WithItemStyleOpts(opts.ItemStyle{Color: colors[i], Opacity: opts.Float(0.3)}),
	)
}
gonb_echarts.Display(line, "width: 1024px; height: 450px; background: white;")

// Print summary table
fmt.Println("\n--- Cumulative Resistant BSI at t=200 (mean ± 2σ across 10 seeds) ---")
for i, key := range policyKeys {
	storages := loadPolicySeeds(key)
	_, mean, std := cumulativeBSIStats(storages)
	last := len(mean) - 1
	fmt.Printf("  %-15s %6.1f ± %5.1f\n", policyNames[i]+":", mean[last], 2*std[last])
}